# Security & Compliance

## AWS IAM Fundamentals

IAM (Identity and Access Management) controls who can access what resources:

```bash
# Create IAM user
aws iam create-user --user-name devops-engineer

# Create access key
aws iam create-access-key --user-name devops-engineer

# Create IAM role
aws iam create-role \
  --role-name ECSTaskRole \
  --assume-role-policy-document file://trust-policy.json

# Attach policy to role
aws iam attach-role-policy \
  --role-name ECSTaskRole \
  --policy-arn arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess

# Create inline policy
aws iam put-role-policy \
  --role-name ECSTaskRole \
  --policy-name S3Access \
  --policy-document file://policy.json
```

## IAM Policies

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "s3:GetObject",
        "s3:ListBucket"
      ],
      "Resource": [
        "arn:aws:s3:::my-bucket",
        "arn:aws:s3:::my-bucket/*"
      ]
    },
    {
      "Effect": "Allow",
      "Action": "dynamodb:Query",
      "Resource": "arn:aws:dynamodb:us-east-1:123456789:table/Users",
      "Condition": {
        "StringEquals": {
          "aws:RequestedRegion": "us-east-1"
        }
      }
    },
    {
      "Effect": "Deny",
      "Action": "ec2:TerminateInstances",
      "Resource": "*"
    }
  ]
}
```

## AWS Secrets Manager

Secrets Manager securely stores and rotates secrets:

```bash
# Create secret
aws secretsmanager create-secret \
  --name prod/db-password \
  --secret-string '{"username":"admin","password":"SecurePassword123"}'

# Get secret
aws secretsmanager get-secret-value \
  --secret-id prod/db-password

# Update secret
aws secretsmanager update-secret \
  --secret-id prod/db-password \
  --secret-string '{"username":"admin","password":"NewPassword456"}'

# Rotate secret
aws secretsmanager rotate-secret \
  --secret-id prod/db-password \
  --rotation-rules AutomaticallyAfterDays=30
```

Use secrets in applications:

In [ ]:
import boto3
import json

secrets_client = boto3.client('secretsmanager')

def get_db_credentials():
    response = secrets_client.get_secret_value(
        SecretId='prod/db-password'
    )
    secret = json.loads(response['SecretString'])
    return secret['username'], secret['password']

username, password = get_db_credentials()

## AWS GuardDuty

GuardDuty detects threats using machine learning:

```bash
# Enable GuardDuty
aws guardduty create-detector --enable

# List detectors
aws guardduty list-detectors

# Get findings
aws guardduty list-findings \
  --detector-id 12abc34d567e8fa901bc2d3e4f5a6b78

# Get finding details
aws guardduty get-findings \
  --detector-id 12abc34d567e8fa901bc2d3e4f5a6b78 \
  --finding-ids arn:aws:guardduty:us-east-1:123456789:detector/12abc34d567e8fa901bc2d3e4f5a6b78/finding/abc123def456
```

## VPC Security

```bash
# Create security group
aws ec2 create-security-group \
  --group-name app-sg \
  --description "Security group for application" \
  --vpc-id vpc-12345

# Allow inbound traffic
aws ec2 authorize-security-group-ingress \
  --group-id sg-12345 \
  --protocol tcp \
  --port 443 \
  --cidr 0.0.0.0/0

# Deny all outbound traffic except HTTPS
aws ec2 revoke-security-group-egress \
  --group-id sg-12345 \
  --protocol -1 \
  --cidr 0.0.0.0/0

aws ec2 authorize-security-group-egress \
  --group-id sg-12345 \
  --protocol tcp \
  --port 443 \
  --cidr 0.0.0.0/0
```

## Network ACLs

```bash
# Create network ACL
aws ec2 create-network-acl --vpc-id vpc-12345

# Add inbound rule
aws ec2 create-network-acl-entry \
  --network-acl-id acl-12345 \
  --rule-number 100 \
  --protocol tcp \
  --port-range From=443,To=443 \
  --cidr-block 0.0.0.0/0 \
  --ingress

# Add outbound rule
aws ec2 create-network-acl-entry \
  --network-acl-id acl-12345 \
  --rule-number 100 \
  --protocol tcp \
  --port-range From=443,To=443 \
  --cidr-block 0.0.0.0/0 \
  --egress
```

## Compliance as Code

```hcl
# Terraform for security compliance
resource "aws_s3_bucket" "secure" {
  bucket = "secure-bucket"
}

# Enable versioning
resource "aws_s3_bucket_versioning" "secure" {
  bucket = aws_s3_bucket.secure.id
  versioning_configuration {
    status = "Enabled"
  }
}

# Enable encryption
resource "aws_s3_bucket_server_side_encryption_configuration" "secure" {
  bucket = aws_s3_bucket.secure.id

  rule {
    apply_server_side_encryption_by_default {
      sse_algorithm = "AES256"
    }
  }
}

# Block public access
resource "aws_s3_bucket_public_access_block" "secure" {
  bucket = aws_s3_bucket.secure.id

  block_public_acls       = true
  block_public_policy     = true
  ignore_public_acls      = true
  restrict_public_buckets = true
}

# Enable logging
resource "aws_s3_bucket_logging" "secure" {
  bucket = aws_s3_bucket.secure.id

  target_bucket = aws_s3_bucket.logs.id
  target_prefix = "s3-logs/"
}
```

## Encryption

```bash
# Create KMS key
aws kms create-key \
  --description "Key for application encryption"

# Create alias
aws kms create-alias \
  --alias-name alias/app-key \
  --target-key-id arn:aws:kms:us-east-1:123456789:key/12345678-1234-1234-1234-123456789012

# Encrypt data
aws kms encrypt \
  --key-id alias/app-key \
  --plaintext "sensitive data"

# Decrypt data
aws kms decrypt \
  --ciphertext-blob fileb://encrypted-data
```

## Audit Logging

```bash
# Enable CloudTrail
aws cloudtrail create-trail \
  --name production-trail \
  --s3-bucket-name cloudtrail-logs

# Start logging
aws cloudtrail start-logging --trail-name production-trail

# Get events
aws cloudtrail lookup-events \
  --lookup-attributes AttributeKey=ResourceName,AttributeValue=my-instance \
  --max-results 10
```

## Security Best Practices

```yaml
# Pod Security Policy for Kubernetes
apiVersion: policy/v1beta1
kind: PodSecurityPolicy
metadata:
  name: restricted
spec:
  privileged: false
  allowPrivilegeEscalation: false
  requiredDropCapabilities:
    - ALL
  volumes:
    - 'configMap'
    - 'emptyDir'
    - 'projected'
    - 'secret'
    - 'downwardAPI'
    - 'persistentVolumeClaim'
  hostNetwork: false
  hostIPC: false
  hostPID: false
  runAsUser:
    rule: 'MustRunAsNonRoot'
  seLinux:
    rule: 'MustRunAs'
    seLinuxOptions:
      level: "s0:c123,c456"
  fsGroup:
    rule: 'MustRunAs'
    ranges:
      - min: 1
        max: 65535
  readOnlyRootFilesystem: false
```

## Vulnerability Scanning

```bash
# Scan Docker image with Trivy
trivy image myapp:latest

# Scan Kubernetes manifests
trivy config ./k8s/

# Scan filesystem
trivy fs ./src/

# Generate SBOM (Software Bill of Materials)
trivy image --format cyclonedx myapp:latest > sbom.json
```

---

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is AWS IAM used for?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="0">
      <span>Managing EC2 instances</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="1">
      <span>Controlling access to AWS resources</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="2">
      <span>Monitoring application performance</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="3">
      <span>Managing DNS records</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What is AWS Secrets Manager used for?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374629" value="0">
      <span>Securely storing and rotating secrets</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374629" value="1">
      <span>Managing IAM policies</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374629" value="2">
      <span>Encrypting S3 buckets</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374629" value="3">
      <span>Monitoring security events</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ What does AWS GuardDuty do?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284736" value="0">
      <span>Manages firewall rules</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284736" value="1">
      <span>Encrypts data at rest</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284736" value="2">
      <span>Detects threats using machine learning</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284736" value="3">
      <span>Manages SSL certificates</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is the principle of least privilege?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7463829" value="0">
      <span>Grant all permissions to all users</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7463829" value="1">
      <span>Grant only the minimum permissions needed</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7463829" value="2">
      <span>Deny all access by default</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7463829" value="3">
      <span>Use the same password for all accounts</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What is CloudTrail used for?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374629" value="0">
      <span>Audit logging of AWS API calls</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374629" value="1">
      <span>Managing network traffic</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374629" value="2">
      <span>Encrypting data in transit</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374629" value="3">
      <span>Managing SSL certificates</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>